# 02. Jev 로 도구 선택하기

## 배경

도구가 늘어나면 LLM 의 도구 선택이 흔들립니다. 이름이 비슷한 도구를 헷갈리고, 스키마가 전부 프롬프트에 들어가서 토큰도 늘어납니다. 그런데 도구 호출은 사실 두 가지 일입니다.

1. **어떤 도구를 쓸지 고르기** → option 중 하나를 고르는 판단
2. **그 도구의 인자를 채우기** → 문장에서 값을 뽑아 쓰는 생성

1번은 Jev 가 잘하는 일이고 2번은 Jev 가 못 하는 일입니다. 그래서 둘을 나눕니다.

## 직관

식당으로 비유하면 Jev 는 주문을 듣고 "이건 그릴 담당"이라고 넘겨 주는 사람이고, LLM 은 실제로 요리하는 사람입니다. LLM 앞에 도구 10개를 다 늘어놓는 대신 Jev 가 고른 2개만 올려 둡니다.

```
사용자 요청 ──▶ Jev choice (도구 10개 중 선택) ──▶ 상위 2개만 LLM 에 노출 ──▶ LLM 이 인자 작성 ──▶ 도구 실행
```

In [1]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트의 jev_agent 패키지를 불러오기 위함

from dotenv import load_dotenv
load_dotenv("../.env")

from jev_agent.jev import JevClient, choice, noul, score

jev = JevClient()

## 샘플 도구

쇼핑몰 '테디마켓' 고객지원 도구 10개입니다. `search_order` 와 `track_shipping`, `cancel_order` 와 `request_refund` 처럼 헷갈리는 쌍을 일부러 넣었습니다.

In [2]:
from jev_agent.tools import tool_catalog

catalog = tool_catalog()
for name, description in catalog.items():
    print(f"{name:20s} {description}")

search_order         주문 번호로 주문 상세(상품, 결제 금액, 주문 상태)를 조회한다. 무엇을 주문했는지 확인할 때 사용한다.
track_shipping       주문의 현재 배송 위치와 도착 예정일을 조회한다. 물건이 어디쯤인지, 언제 오는지 물을 때 사용한다.
cancel_order         아직 출고되지 않은 주문을 취소한다. 되돌릴 수 없는 작업이다.
request_refund       수령한 상품의 환불을 접수한다. 결제 금액이 고객에게 반환되는 되돌릴 수 없는 작업이다.
change_address       출고 전 주문의 배송지를 변경한다.
check_points         고객의 적립금(포인트) 잔액을 조회한다.
search_products      상품 이름으로 판매 중인 상품과 가격을 검색한다. 구매 전 상품을 찾을 때 사용한다.
check_stock          상품 ID 로 현재 재고 수량을 조회한다. 품절 여부를 물을 때 사용한다.
search_faq           환불 규정, 배송 정책, 쿠폰 사용법 같은 일반 정책 안내문을 검색한다. 특정 주문과 무관한 질문에 사용한다.
escalate_to_human    상담원 연결을 요청한다. 사용자가 사람과 통화하길 원하거나 불만이 심할 때 사용한다.


## 도구 설명을 그대로 choice option 으로 쓰기

`@tool` 의 docstring 이 곧 Jev 의 option 설명이 됩니다. 도구가 필요 없는 경우를 위해 `no_tool` option 을 하나 더 둡니다.

In [3]:
import pandas as pd
from jev_agent.middleware import NO_TOOL

OPTIONS = {**catalog, NO_TOOL: "도구가 필요 없다. 인사나 잡담이다."}

def pick_tool(user_request: str):
    result = jev.decide(
        {"user_request": user_request},
        {"tool": choice("사용자 요청을 처리하기 위해 실행해야 할 도구는 무엇인가?", OPTIONS)},
    )
    return result["tool"], result.latency_ms

queries = [
    "A1001 지금 어디쯤 왔어요?",
    "A1002 제가 뭘 샀었죠?",
    "A1002 받았는데 소리가 안 나요. 돈 돌려주세요.",
    "P20 지금 살 수 있나요?",
    "안녕하세요!",
]
rows = []
for query in queries:
    answer, latency_ms = pick_tool(query)
    top3 = sorted(answer["probabilities"].items(), key=lambda item: -item[1])[:3]
    rows.append({
        "질의": query,
        "선택": answer["choice"],
        "confidence": round(answer["confidence"], 2),
        "상위 3개": ", ".join(f"{name} {p:.2f}" for name, p in top3),
        "지연(ms)": round(latency_ms),
    })
pd.DataFrame(rows)

,질의,선택,confidence,상위 3개,지연(ms)
0,A1001 지금 어디쯤 왔어요?,track_shipping,0.99,"track_shipping 1.00, no_tool 0.00, search_faq ...",242
1,A1002 제가 뭘 샀었죠?,search_order,1.00,"search_order 1.00, check_stock 0.00, search_pr...",265
2,A1002 받았는데 소리가 안 나요. 돈 돌려주세요.,request_refund,0.67,"request_refund 0.70, search_order 0.29, escala...",297
3,P20 지금 살 수 있나요?,check_stock,0.69,"check_stock 0.73, search_products 0.26, check_...",260
4,안녕하세요!,no_tool,1.00,"no_tool 1.00, change_address 0.00, search_faq ...",281


## DeepAgents 미들웨어로 끼워 넣기

DeepAgents 의 에이전트는 LangGraph 그래프이고, 모델을 부르기 직전에 요청을 고칠 수 있는 `wrap_model_call` 훅을 제공합니다. `JevToolSelectorMiddleware` 는 이 훅에서 세 가지를 합니다.

1. 마지막 사용자 메시지와 이미 실행한 도구 이름을 `state` 로 Jev 에 보냅니다.
2. probability 상위 `top_k` 개 도구만 남기고 나머지 커스텀 도구를 요청에서 뺍니다. DeepAgents 내장 도구는 건드리지 않습니다. 내장 파일 도구까지 숨기려면 `hide_builtin_tools=True` 를 줍니다. 샘플 고객지원 에이전트는 이 옵션을 켰습니다. 파일 도구가 보이면 LLM 이 주문 조회 중에 `grep` 을 부르는 일이 있었기 때문입니다.
3. 판단 기록을 그래프 state 의 `jev_decisions` 에 남깁니다.

Jev 에 도구 결과 원문을 넘기지 않고 **실행한 도구 이름만** 넘기는 데는 이유가 있습니다. 결과 원문이 state 에 남아 있으면 Jev 가 같은 도구를 다시 고르는 경향이 있기 때문입니다.

In [4]:
import inspect
from jev_agent.middleware import JevToolSelectorMiddleware

print(inspect.getsource(JevToolSelectorMiddleware._narrow))

    def _narrow(self, request: Any, result: JevResult | None) -> tuple[Any, dict[str, Any]]:
        if result is None:
            return request, record("tool_select", "도구 선택", "Jev 실패 → LLM 에 전체 위임", None)
        answer = result["tool"]
        confidence = answer.get("confidence", 0.0)
        probabilities = answer.get("probabilities", {})
        if confidence < self.min_confidence:
            entry = record(
                "tool_select",
                "도구 선택",
                "확신 낮음 → LLM 에 전체 위임",
                result,
                probabilities=probabilities,
                confidence=confidence,
            )
            return request, entry
        ranked = sorted(probabilities, key=probabilities.get, reverse=True)[: self.top_k]
        allowed = {
            name
            for name in ranked
            if name != NO_TOOL and probabilities[name] >= self.min_probability
        }
        # Jev 가 관리하는 도구(catalog)만 걸러 낸다. write_todos 같은 내장 도구는 그대로 둔다.
        to

In [5]:
from deepagents import create_deep_agent
from jev_agent.agent import SUPPORT_SYSTEM_PROMPT, build_chat_model
from jev_agent.tools import TOOLS

agent = create_deep_agent(
    model=build_chat_model(),
    tools=TOOLS,
    system_prompt=SUPPORT_SYSTEM_PROMPT,
    middleware=[JevToolSelectorMiddleware(jev, catalog=catalog, top_k=2)],
)

state = agent.invoke({"messages": [{"role": "user", "content": "A1001 지금 어디쯤 왔어요?"}]})

for message in state["messages"]:
    calls = [call["name"] for call in getattr(message, "tool_calls", [])]
    print(f"[{message.type}] {message.content or calls}")

[human] A1001 지금 어디쯤 왔어요?
[ai] ['track_shipping']
[tool] 주문 A1001: 옥천 허브 통과, 내일 도착 예정 (서울시 마포구)
[ai] 주문 A1001은 현재 옥천 허브를 통과했고, 내일 서울시 마포구로 도착 예정입니다.


`jev_decisions` 를 보면 모델 호출마다 Jev 가 무엇을 골랐는지 남아 있습니다. 두 번째 호출에서는 필요한 도구가 이미 실행됐으므로 `no_tool` 쪽 probability 가 올라가는지 확인해 보세요.

In [6]:
for decision in state["jev_decisions"]:
    print(decision["verdict"], "| confidence", decision.get("confidence"), "|", decision["latency_ms"], "ms")

track_shipping | confidence 0.98 | 249 ms
track_shipping | confidence 0.69 | 282 ms


## 정리

- 도구 선택은 Jev 의 `choice`, 인자 작성은 LLM 으로 나눴습니다.
- LLM 은 좁혀진 후보만 보므로 헷갈릴 option 이 줄고 프롬프트도 짧아집니다.
- Jev 는 option 밖의 답을 낼 수 없어서 없는 도구 이름이 나오지 않습니다.

실제로 정확도가 얼마나 달라지는지는 05번에서 측정합니다. 그 전에, Jev 가 확신하지 못할 때 어떻게 할지를 03번에서 다룹니다.